# OPEN API를 활용한 네이버 뉴스 검핵

## 1. 애플리케이션 등록
https://developers.naver.com/apps/#/register

## 2. 환경변수 관리
- 등록된 애플리케이션 페이지에서 제공되는 Client ID, Secret은 절대 외부 노출 금지
- dotenv (.ent)를 통해서 관리
    - `pip install python-dotenv` 패키지설치
    - .git ignore 파일에 .env 무시하는 구문 추가



In [20]:
!pip install python-dotenv

### 프로젝트 하위 폴더에 .env 파일을 만들어 아래 값 입력

```
NAVER_CLIENT_DI={{Client Id}}
NAVER_CLIENT_SECRET={{Client Secret}}
```
- 네이버 개발자 센터에 등록된 애플리케이션에서 확인 가능

In [1]:
# .env 파일을 로드해서 환경변수로 등록
from dotenv import load_dotenv

load_dotenv() # .env 파일을 읽어와 자동으로 환경변수로 등록
# 읽어오기 성공 시 True, 실패 시 False


True

In [2]:
# 환경변수에서 .env 등록한 내용 얻어오기
import os
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

# api key, 비번을 print하는 구문을 실수로 남겨놓으면 노출될 가능성 있음
# -> jupyter 변수 탭에서 확인

# .env 내용이 환경변수로 등록되지 않은 경우
# if not NAVER_CLIENT_SECRET or not NAVER_CLIENT_ID:
#     raise ValueError('네이버 클라이언트 아이디 또는 시크릿 키가 정의되지 않았습니다')

In [23]:
!pip install requests

In [3]:
# 네이버 뉴스 검색 API 요청
import urllib.request
import socket

encText = urllib.parse.quote('인공지능')
# 한글 -> %시작하는 문자로 변경

url = f'https://openapi.naver.com/v1/search/news.json?query={encText}&display=10&sort=date'

# 요청객체 생성
request = urllib.request.Request(url)

#API인증정보를 요청
request.add_header('X-Naver-Client-Id', NAVER_CLIENT_ID)
request.add_header('X-Naver-Client-Secret', NAVER_CLIENT_SECRET)

try:
    with urllib.request.urlopen(request, timeout=10) as response:
        # 지정된 주소로 요청하고 결과를 response로 받음
        # 단, 요청 대기 시간이 10초를 경과하면 중지

        # http응답 상태코드
        # 200 -> 성공

        response_code = response.getcode()
        response_body = response.read().decode('utf-8')
        print("response_code: ", response_code)
        print(response_body)

        # 응답 본문이 JSON

except socket.timeout:
    print("요청 시간 10초 초과")




response_code:  200
{
	"lastBuildDate":"Mon, 15 Jun 2026 11:27:29 +0900",
	"total":4045665,
	"start":1,
	"display":10,
	"items":[
		{
			"title":"보수의 오적(五賊)[이용식의 시론]",
			"originallink":"https:\/\/www.munhwa.com\/article\/11595552?ref=naver",
			"link":"https:\/\/n.news.naver.com\/mnews\/article\/021\/0002797731?sid=110",
			"description":"한동훈·이준석 등과 ‘빅 텐트’를 펼치고, 정책은 <b>인공지능<\/b>(AI) 시대에 맞춰야 한다. 윈스턴 처칠도, 박근혜 전 대통령도 한때 보수 정당을 떠났던 적이 있다. K양극화에 대응해 ‘스마트 분배’ 등 정책 대전환도... ",
			"pubDate":"Mon, 15 Jun 2026 11:26:00 +0900"
		},
		{
			"title":"[창간71주년 특집] 지방대학병원의 경쟁력 ⑦",
			"originallink":"http:\/\/www.whosaeng.com\/171303",
			"link":"http:\/\/www.whosaeng.com\/171303",
			"description":"특히 <b>인공지능<\/b>(AI) 기반 의료데이터 분석과 디지털 헬스케어 기술 개발, 바이오 빅데이터 연구 등이 활발히 진행되고 있다. 의료현장에서 축적되는 임상데이터를 활용해 질병 예측과 맞춤형 치료기술 개발에 나서며... ",
			"pubDate":"Mon, 15 Jun 2026 11:26:00 +0900"
		},
		{
			"title":"[창간71주년 특집] 지방대학병원의 경쟁력 ④",
			"originallink":"http:\/\/www.whosaeng.com\/171306",
			"link":"http:\/\/www.whos

In [7]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.json'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.json() # 응답 데이터(json) -> dict 변환

    print('response_code: ', response_code)
    # print(data)
    pprint(data['items'][0])
except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

response_code:  200
{'description': '이번 명단에는 전자상거래 기업 알리바바, 검색·<b>인공지능</b>(AI) 기업 바이두, 전기차 기업 BYD와 '
                '니오, 태양광 기업 트리나솔라와 JA솔라테크놀로지 등이 포함됐다. 이 명단은 미국 국방수권법에 근거한 이른바 '
                '1260H 명단이다.... ',
 'link': 'https://www.hellot.net/news/article.html?no=113187',
 'originallink': 'https://www.hellot.net/news/article.html?no=113187',
 'pubDate': 'Mon, 15 Jun 2026 11:48:00 +0900',
 'title': '美, 알리바바·바이두·BYD 군사기업 명단 추가…中 반발'}


In [12]:
# requests 객체를 이용한 요청(더 쉬움)
import requests
from pprint import pprint # 출력 시 공백문자를 이용해 가독성 좋게 출력

url = 'https://openapi.naver.com/v1/search/news.xml'

# Header, Body에 전달할 값을 dict 형식으로 생성
headers = {
    "X-Naver-Client-id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

params = {
    'query' : '인공지능',
    'display' : 10,
    'start' : 1,
    'sort' : 'date'
}

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,
        headers=headers,
        params=params, # dict -> 쿼리 스트링 변환(+url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대 인 경우 예외 발생
    response.raise_for_status()

    response_code = response.status_code # 상태 코드
    data = response.content # 응답을 문자열로 변경

    # xml 해석 파이썬 기본 내장 모듈
    # -> xml.etree.ElementTree


    with open('response.xml', 'wb')as f:
        f.write(data)

    print('response.xml 저장완료')
    pprint(data['items'][0])
except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except Exception as e:
    print(e)

response.xml 저장완료
byte indices must be integers or slices, not str
